# Preparación del entorno
Importamos las librerias necesarias como primer paso. 

In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\XxDEs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\XxDEs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\XxDEs\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Carga y limpieza de los datos
Cargamos los datos y nos quedamos con las columnas que nos interesan y las filas que tengan sinopsis. 

In [5]:
ruta_dataset = './tmdb_5000_movies.csv'
df_movies = pd.read_csv(ruta_dataset)

df_movies = df_movies[['title', 'overview']]

df_movies = df_movies.dropna(subset=['overview'])

df_movies = df_movies.reset_index(drop=True)

print(f"Total de películas tras limpieza: {df_movies.shape[0]}")
df_movies.head(3)

Total de películas tras limpieza: 4800


,title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...


# Preprocesamiento del texto
Aplicamos las técnicas necesarias a la columna de sinopsis. 

In [6]:
stop_words = set(stopwords.words('english'))

def preprocesar_sinopsis(texto):
    texto = texto.lower()

    texto = re.sub(r'[^a-z\s]', '', texto)
    
    tokens = word_tokenize(texto)
    
    tokens_procesados = [word for word in tokens if word not in stop_words]
    
    return tokens_procesados

df_movies['tokens_overview'] = df_movies['overview'].apply(preprocesar_sinopsis)

df_movies[['title', 'tokens_overview']].head(3)

,title,tokens_overview
0,Avatar,"[nd, century, paraplegic, marine, dispatched, ..."
1,Pirates of the Caribbean: At World's End,"[captain, barbossa, long, believed, dead, come..."
2,Spectre,"[cryptic, message, bonds, past, sends, trail, ..."


# Representación de las películas (Word Embeddings)
Se utiliza Gensim para descargar un modelo preentrenado de GloVe y se convierte cada sinopsis en el promedio de los vectores de sus palabras. 

In [7]:
print("Cargando modelo GloVe...")
modelo_glove = api.load("glove-wiki-gigaword-50")
print("¡Modelo cargado!")

def obtener_vector_sinopsis(tokens, modelo):
    vectores = []
    for palabra in tokens:
        if palabra in modelo:
            vectores.append(modelo[palabra])
    
    if len(vectores) == 0:
        return np.zeros(modelo.vector_size)
    
    return np.mean(vectores, axis=0)

matriz_embeddings = np.array([obtener_vector_sinopsis(tokens, modelo_glove) for tokens in df_movies['tokens_overview']])

print(f"Forma de la matriz de embeddings: {matriz_embeddings.shape}")

Cargando modelo GloVe...
[==================================================] 100.0% 66.0/66.0MB downloaded
¡Modelo cargado!
Forma de la matriz de embeddings: (4800, 50)


# Cálculo de similitud y sistema de recomendación
Aquí calcularemos la similitud del coseno e implementaremos la función principal para obtener las 10 recomendaciones.

In [8]:
similitud_matriz = cosine_similarity(matriz_embeddings)

def recomendar_peliculas(titulo_pelicula, df, matriz_similitud, top_n=10):
    if titulo_pelicula not in df['title'].values:
        return f"La película '{titulo_pelicula}' no se encuentra en la base de datos."
    
    idx = df[df['title'] == titulo_pelicula].index[0]
    
    puntuaciones = list(enumerate(matriz_similitud[idx]))
    
    puntuaciones = sorted(puntuaciones, key=lambda x: x[1], reverse=True)
    
    puntuaciones_top = puntuaciones[1:top_n+1]
    
    indices_recomendados = [i[0] for i in puntuaciones_top]
    titulos_recomendados = df['title'].iloc[indices_recomendados].tolist()
    
    return titulos_recomendados


print("Recomendaciones para 'The Dark Knight Rises':")
for i, peli in enumerate(recomendar_peliculas('The Dark Knight Rises', df_movies, similitud_matriz)):
    print(f"{i+1}. {peli}")

print("\n" + "-"*40 + "\n")

print("Recomendaciones para 'Interstellar':")
for i, peli in enumerate(recomendar_peliculas('Interstellar', df_movies, similitud_matriz)):
    print(f"{i+1}. {peli}")

Recomendaciones para 'The Dark Knight Rises':
1. The Manchurian Candidate
2. The Dark Knight
3. Max Payne
4. The Glimmer Man
5. Rockaway
6. Sea of Love
7. Street Kings
8. R.I.P.D.
9. Red Riding: In the Year of Our Lord 1974
10. Blood Done Sign My Name

----------------------------------------

Recomendaciones para 'Interstellar':
1. Journey to Saturn
2. The Cave
3. Silent Running
4. 2001: A Space Odyssey
5. Lifeforce
6. Antarctic Edge: 70° South
7. Spy Kids 2: The Island of Lost Dreams
8. Capricorn One
9. Sea Rex 3D: Journey to a Prehistoric World
10. Space Battleship Yamato
